# The Skill is the Unit of Reuse

Companion notebook for the blog post on skills in AI agentic systems.

This notebook demonstrates:
- Building and parsing SKILL.md files
- Progressive disclosure (two-phase skill loading)
- A multi-skill router using the Claude API
- Skill evaluation with activation precision/recall

**Note:** API calls require an `ANTHROPIC_API_KEY` environment variable. Cells that call the API are marked and can be skipped — the parsing and evaluation logic works without an API key.

In [1]:
!pip install pyyaml anthropic


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python3.12 -m pip install --upgrade pip


In [2]:
import os
import json
import yaml
import tempfile
from pathlib import Path
from typing import Optional

## 1. Anatomy of a Skill

A skill is a folder with a `SKILL.md` file. Let's create a realistic example — a deploy-review skill — and parse it.

In [3]:
# Create a temporary skills directory with two example skills
skills_dir = tempfile.mkdtemp(prefix="skills_")

# Skill 1: deploy-review
deploy_skill_dir = os.path.join(skills_dir, "deploy-review")
os.makedirs(deploy_skill_dir)

deploy_skill_md = """---
name: deploy-review
description: >
  Reviews deployment plans for safety and correctness before
  pushing to production. Use when the user mentions deploying,
  releasing, shipping, or pushing to prod.
user-invocable: true
allowed-tools: ["Bash", "Read", "Glob", "Grep"]
---

# Deploy Review

Before any deployment, verify the following:

## Pre-flight Checks

1. Run the test suite and confirm all tests pass
2. Check that the diff doesn't include secrets or credentials
3. Verify database migrations are backward-compatible
4. Confirm the rollback procedure is documented

## Review Process

Read the deployment manifest:
- Compare against the production config
- Flag any new environment variables
- Check resource limits (CPU, memory) against cluster capacity

## Output Format

Respond with exactly this structure:

### Summary
One paragraph describing the overall assessment.

### Issues Found
- **[SEVERITY]** Description of issue (file:line)

### Recommendation
APPROVE / REQUEST_CHANGES / BLOCK
"""

with open(os.path.join(deploy_skill_dir, "SKILL.md"), "w") as f:
    f.write(deploy_skill_md)

print(f"Created deploy-review skill at: {deploy_skill_dir}")

Created deploy-review skill at: /var/folders/3_/j9hs9p_x08sgnll9vzf8zws00000gn/T/skills_ou2d2ibi/deploy-review


In [4]:
# Skill 2: test-runner
test_skill_dir = os.path.join(skills_dir, "test-runner")
os.makedirs(test_skill_dir)

test_skill_md = """---
name: test-runner
description: >
  Runs test suites and reports results with coverage.
  Use when the user asks to run tests, check coverage,
  or verify that code changes don't break anything.
user-invocable: true
allowed-tools: ["Bash", "Read"]
---

# Test Runner

## Steps

1. Detect the test framework (pytest, jest, go test, etc.)
2. Run the full test suite with coverage enabled
3. Parse the output for failures and coverage percentage
4. Report results in a structured format

## Output Format

- **Tests**: X passed, Y failed, Z skipped
- **Coverage**: XX%
- **Failures**: List each failure with file:line and error message
"""

with open(os.path.join(test_skill_dir, "SKILL.md"), "w") as f:
    f.write(test_skill_md)

print(f"Created test-runner skill at: {test_skill_dir}")
print(f"\nSkills directory: {skills_dir}")
print(f"Contents: {os.listdir(skills_dir)}")

Created test-runner skill at: /var/folders/3_/j9hs9p_x08sgnll9vzf8zws00000gn/T/skills_ou2d2ibi/test-runner

Skills directory: /var/folders/3_/j9hs9p_x08sgnll9vzf8zws00000gn/T/skills_ou2d2ibi
Contents: ['deploy-review', 'test-runner']


## 2. Parsing Skills — Frontmatter and Body

Every skill has YAML frontmatter (metadata) and a markdown body (instructions). Let's build a parser.

In [5]:
def parse_skill(skill_path: str) -> dict:
    """Parse a SKILL.md file into frontmatter and body."""
    with open(skill_path) as f:
        content = f.read()

    # Split on YAML frontmatter delimiters
    parts = content.split("---", 2)
    if len(parts) < 3:
        return {"meta": {}, "body": content}

    meta = yaml.safe_load(parts[1])
    body = parts[2].strip()

    return {
        "meta": meta,
        "body": body,
        "body_lines": len(body.splitlines()),
        "body_tokens_approx": len(body.split()),  # Rough estimate
    }


# Parse both skills
deploy_parsed = parse_skill(
    os.path.join(deploy_skill_dir, "SKILL.md")
)
test_parsed = parse_skill(
    os.path.join(test_skill_dir, "SKILL.md")
)

print("=== deploy-review ===")
print(f"Name: {deploy_parsed['meta']['name']}")
print(f"Description: {deploy_parsed['meta']['description'][:80]}...")
print(f"Tools: {deploy_parsed['meta'].get('allowed-tools', 'any')}")
print(f"Body: {deploy_parsed['body_lines']} lines, ~{deploy_parsed['body_tokens_approx']} tokens")
print()
print("=== test-runner ===")
print(f"Name: {test_parsed['meta']['name']}")
print(f"Description: {test_parsed['meta']['description'][:80]}...")
print(f"Body: {test_parsed['body_lines']} lines, ~{test_parsed['body_tokens_approx']} tokens")

=== deploy-review ===
Name: deploy-review
Description: Reviews deployment plans for safety and correctness before pushing to production...
Tools: ['Bash', 'Read', 'Glob', 'Grep']
Body: 30 lines, ~105 tokens

=== test-runner ===
Name: test-runner
Description: Runs test suites and reports results with coverage. Use when the user asks to ru...
Body: 14 lines, ~64 tokens


## 3. Progressive Disclosure — Two-Phase Skill Loading

Phase 1 loads only names and descriptions (~100 tokens per skill).
Phase 2 loads the full SKILL.md for the matched skill.

This keeps token costs low when you have many skills.

In [6]:
def load_skill_registry(skills_dir: str) -> dict:
    """Phase 1: Load name + description for all skills."""
    registry = {}
    for name in os.listdir(skills_dir):
        skill_path = os.path.join(skills_dir, name, "SKILL.md")
        if not os.path.exists(skill_path):
            continue
        parsed = parse_skill(skill_path)
        meta = parsed["meta"]
        registry[meta["name"]] = {
            "description": meta.get("description", ""),
            "path": skill_path,
            # Phase 1 cost: just name + description
            "discovery_tokens": len(meta["name"].split())
                + len(meta.get("description", "").split()),
        }
    return registry


registry = load_skill_registry(skills_dir)

print("=== Skill Registry (Phase 1) ===")
total_discovery_tokens = 0
for name, info in registry.items():
    print(f"\n{name}:")
    print(f"  Description: {info['description'][:60]}...")
    print(f"  Discovery cost: ~{info['discovery_tokens']} tokens")
    total_discovery_tokens += info["discovery_tokens"]

print(f"\nTotal discovery cost for {len(registry)} skills: ~{total_discovery_tokens} tokens")

=== Skill Registry (Phase 1) ===

deploy-review:
  Description: Reviews deployment plans for safety and correctness before p...
  Discovery cost: ~24 tokens

test-runner:
  Description: Runs test suites and reports results with coverage. Use when...
  Discovery cost: ~27 tokens

Total discovery cost for 2 skills: ~51 tokens


In [7]:
def build_routing_prompt(registry: dict) -> str:
    """Build a prompt that lets the model pick a skill."""
    skills_list = "\n".join(
        f"- **{name}**: {info['description']}"
        for name, info in registry.items()
    )
    return f"""You have access to these skills:

{skills_list}

Given the user's request, respond with the skill name
that best matches. If no skill matches, respond with
"none". Respond with ONLY the skill name."""


routing_prompt = build_routing_prompt(registry)
print("=== Routing Prompt ===")
print(routing_prompt)
print(f"\nRouting prompt length: ~{len(routing_prompt.split())} tokens")

=== Routing Prompt ===
You have access to these skills:

- **deploy-review**: Reviews deployment plans for safety and correctness before pushing to production. Use when the user mentions deploying, releasing, shipping, or pushing to prod.

- **test-runner**: Runs test suites and reports results with coverage. Use when the user asks to run tests, check coverage, or verify that code changes don't break anything.


Given the user's request, respond with the skill name
that best matches. If no skill matches, respond with
"none". Respond with ONLY the skill name.

Routing prompt length: ~84 tokens


## 4. Skill Router — Full Implementation

This section combines Phase 1 (routing) and Phase 2 (execution) into a complete skill router.

**Note:** The API calls below require an `ANTHROPIC_API_KEY`. If you don't have one, you can still read the code — the logic is the important part.

In [8]:
def route_to_skill(
    user_input: str,
    registry: dict,
    use_api: bool = False,
) -> Optional[str]:
    """
    Route a user request to the best matching skill.

    If use_api=False, uses keyword matching (for testing).
    If use_api=True, uses Claude Haiku for routing.
    """
    if use_api:
        import anthropic
        client = anthropic.Anthropic()
        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=50,
            system=build_routing_prompt(registry),
            messages=[{"role": "user", "content": user_input}],
        )
        skill_name = response.content[0].text.strip()
    else:
        # Simple keyword matching for offline testing
        user_lower = user_input.lower()
        best_match = None
        best_score = 0
        for name, info in registry.items():
            desc_words = info["description"].lower().split()
            score = sum(
                1 for w in desc_words if w in user_lower
            )
            if score > best_score:
                best_score = score
                best_match = name
        skill_name = best_match if best_score > 2 else "none"

    if skill_name == "none" or skill_name not in registry:
        return None
    return skill_name


# Test routing with keyword matching (no API needed)
test_inputs = [
    "Can you review my deployment plan?",
    "Run the test suite and check coverage",
    "What's the weather today?",
    "Ship this to production please",
    "Are there any test failures?",
]

print("=== Skill Routing (keyword mode) ===")
for inp in test_inputs:
    result = route_to_skill(inp, registry, use_api=False)
    print(f"  '{inp[:50]}...' -> {result or 'none'}")

=== Skill Routing (keyword mode) ===
  'Can you review my deployment plan?...' -> none
  'Run the test suite and check coverage...' -> test-runner
  'What's the weather today?...' -> deploy-review
  'Ship this to production please...' -> none
  'Are there any test failures?...' -> none


In [9]:
def load_full_skill(skill_name: str, registry: dict) -> str:
    """Phase 2: Load the full SKILL.md for execution."""
    if skill_name not in registry:
        raise ValueError(f"Skill '{skill_name}' not found")

    skill_path = registry[skill_name]["path"]
    with open(skill_path) as f:
        content = f.read()

    print(f"Loaded skill '{skill_name}': {len(content)} chars")
    return content


# Demonstrate Phase 2 loading
routed = route_to_skill(
    "Review my deploy plan", registry, use_api=False
)
if routed:
    full_skill = load_full_skill(routed, registry)
    # Show first 200 chars of loaded skill
    print(f"\nFirst 200 chars of loaded skill:\n{full_skill[:200]}...")

## 5. Skill Evaluation

The most important unsolved problem: how do you test a skill?

We need to measure:
1. **Activation accuracy**: Does the skill fire for the right inputs?
2. **Instruction adherence**: Does the output match the expected format?
3. **Edge cases**: What happens with ambiguous or adversarial inputs?

Let's build an eval framework.

In [10]:
# Create an evals.json for the deploy-review skill
deploy_evals = {
    "skill": "deploy-review",
    "evals": [
        {
            "input": "Review the deployment plan in deploy.yaml",
            "expected_activation": True,
            "expected_output_contains": ["Status:", "Risks:"],
            "expected_tools_used": ["Read", "Bash"],
        },
        {
            "input": "Can you check if this is safe to ship to prod?",
            "expected_activation": True,
        },
        {
            "input": "Deploy this to staging",
            "expected_activation": True,
        },
        {
            "input": "What's the weather today?",
            "expected_activation": False,
        },
        {
            "input": "Write a unit test for the login page",
            "expected_activation": False,
        },
        {
            "input": "Run the tests",
            "expected_activation": False,
        },
        {
            "input": "Ship it",
            "expected_activation": True,
        },
        {
            "input": "Is the release pipeline green?",
            "expected_activation": True,
        },
    ],
}

# Save alongside the skill
evals_path = os.path.join(deploy_skill_dir, "evals.json")
with open(evals_path, "w") as f:
    json.dump(deploy_evals, f, indent=2)

print(f"Created evals.json with {len(deploy_evals['evals'])} test cases")
print(f"  - {sum(1 for e in deploy_evals['evals'] if e['expected_activation'])} should activate")
print(f"  - {sum(1 for e in deploy_evals['evals'] if not e['expected_activation'])} should NOT activate")

Created evals.json with 8 test cases
  - 5 should activate
  - 3 should NOT activate


In [11]:
def eval_skill_activation(
    skill_name: str,
    evals: dict,
    registry: dict,
    use_api: bool = False,
) -> dict:
    """Evaluate skill activation precision and recall."""
    results = {
        "true_pos": 0,   # Should activate, did activate
        "false_pos": 0,  # Should NOT activate, did activate
        "true_neg": 0,   # Should NOT activate, didn't activate
        "false_neg": 0,  # Should activate, didn't activate
        "details": [],
    }

    for case in evals["evals"]:
        routed = route_to_skill(
            case["input"], registry, use_api=use_api
        )
        activated = routed == skill_name
        expected = case["expected_activation"]

        if activated and expected:
            results["true_pos"] += 1
            status = "TP"
        elif activated and not expected:
            results["false_pos"] += 1
            status = "FP"
        elif not activated and not expected:
            results["true_neg"] += 1
            status = "TN"
        else:
            results["false_neg"] += 1
            status = "FN"

        results["details"].append({
            "input": case["input"][:50],
            "expected": expected,
            "actual": activated,
            "routed_to": routed,
            "status": status,
        })

    # Calculate precision and recall
    tp = results["true_pos"]
    fp = results["false_pos"]
    fn = results["false_neg"]

    results["precision"] = tp / (tp + fp) if (tp + fp) > 0 else 0
    results["recall"] = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1_denom = results["precision"] + results["recall"]
    results["f1"] = (
        2 * results["precision"] * results["recall"] / f1_denom
        if f1_denom > 0 else 0
    )

    return results


# Run evaluation
eval_results = eval_skill_activation(
    "deploy-review", deploy_evals, registry, use_api=False
)

print("=== Activation Evaluation ===")
print(f"Precision: {eval_results['precision']:.2f}")
print(f"Recall:    {eval_results['recall']:.2f}")
print(f"F1 Score:  {eval_results['f1']:.2f}")
print(f"\nConfusion matrix:")
print(f"  TP={eval_results['true_pos']}  FP={eval_results['false_pos']}")
print(f"  FN={eval_results['false_neg']}  TN={eval_results['true_neg']}")
print(f"\nDetails:")
for d in eval_results["details"]:
    icon = "pass" if d["status"] in ("TP", "TN") else "FAIL"
    print(f"  [{icon}] [{d['status']}] '{d['input']}' -> {d['routed_to'] or 'none'}")

=== Activation Evaluation ===
Precision: 0.00
Recall:    0.00
F1 Score:  0.00

Confusion matrix:
  TP=0  FP=2
  FN=5  TN=1

Details:
  [FAIL] [FN] 'Review the deployment plan in deploy.yaml' -> none
  [FAIL] [FN] 'Can you check if this is safe to ship to prod?' -> none
  [FAIL] [FN] 'Deploy this to staging' -> none
  [FAIL] [FP] 'What's the weather today?' -> deploy-review
  [FAIL] [FP] 'Write a unit test for the login page' -> deploy-review
  [pass] [TN] 'Run the tests' -> test-runner
  [FAIL] [FN] 'Ship it' -> none
  [FAIL] [FN] 'Is the release pipeline green?' -> none


## 6. Cross-Platform Skill Compatibility

The Agent Skills spec means the same SKILL.md works across Claude Code, Mistral Vibe, and any tool that adopts the standard. Let's verify our skill is spec-compliant.

In [12]:
import re

def validate_skill(skill_path: str) -> list:
    """Validate a SKILL.md against the Agent Skills spec."""
    issues = []
    parsed = parse_skill(skill_path)
    meta = parsed["meta"]

    # Required fields
    if "name" not in meta:
        issues.append("MISSING: 'name' field is required")
    elif not re.match(r"^[a-z0-9-]+$", meta["name"]):
        issues.append(
            f"INVALID: name '{meta['name']}' must be "
            f"lowercase letters, numbers, hyphens only"
        )
    elif len(meta["name"]) > 64:
        issues.append(
            f"INVALID: name is {len(meta['name'])} chars "
            f"(max 64)"
        )

    if "description" not in meta:
        issues.append("MISSING: 'description' field is required")
    elif len(meta["description"]) > 1024:
        issues.append(
            f"INVALID: description is {len(meta['description'])} "
            f"chars (max 1024)"
        )

    # Best practices (warnings, not errors)
    if parsed["body_lines"] > 500:
        issues.append(
            f"WARNING: body is {parsed['body_lines']} lines "
            f"(recommended max 500)"
        )

    desc = meta.get("description", "")
    if len(desc.split()) < 10:
        issues.append(
            "WARNING: description is very short — include "
            "trigger words for better activation"
        )

    return issues


# Validate our skills
for name in os.listdir(skills_dir):
    skill_path = os.path.join(skills_dir, name, "SKILL.md")
    if os.path.exists(skill_path):
        issues = validate_skill(skill_path)
        status = "VALID" if not issues else f"{len(issues)} issue(s)"
        print(f"\n{name}: {status}")
        for issue in issues:
            print(f"  - {issue}")
        if not issues:
            print("  All checks passed!")


deploy-review: VALID
  All checks passed!

test-runner: VALID
  All checks passed!


## 7. Putting It All Together

Here's the complete flow: discover skills, route a request, load the matched skill, and (optionally) execute via the Claude API.

In [13]:
def skill_pipeline(
    user_request: str,
    skills_dir: str,
    use_api: bool = False,
) -> dict:
    """Complete skill pipeline: discover -> route -> load."""
    # Phase 1: Discover
    registry = load_skill_registry(skills_dir)
    print(f"Discovered {len(registry)} skills")

    # Route
    skill_name = route_to_skill(
        user_request, registry, use_api=use_api
    )
    if not skill_name:
        return {"status": "no_match", "skill": None}

    print(f"Routed to: {skill_name}")

    # Phase 2: Load full skill
    full_skill = load_full_skill(skill_name, registry)

    # If API is available, execute
    if use_api:
        import anthropic
        client = anthropic.Anthropic()
        response = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=4096,
            system=full_skill,
            messages=[{
                "role": "user",
                "content": user_request,
            }],
        )
        return {
            "status": "executed",
            "skill": skill_name,
            "response": response.content[0].text,
        }

    return {
        "status": "routed",
        "skill": skill_name,
        "skill_tokens": len(full_skill.split()),
    }


# Demo the pipeline
result = skill_pipeline(
    "Review my deployment before we push to production",
    skills_dir,
    use_api=False,
)
print(f"\nResult: {json.dumps(result, indent=2)}")

Discovered 2 skills
Routed to: deploy-review
Loaded skill 'deploy-review': 1008 chars

Result: {
  "status": "routed",
  "skill": "deploy-review",
  "skill_tokens": 141
}


## Cleanup

In [14]:
import shutil
shutil.rmtree(skills_dir)
print(f"Cleaned up temporary skills directory: {skills_dir}")

Cleaned up temporary skills directory: /var/folders/3_/j9hs9p_x08sgnll9vzf8zws00000gn/T/skills_ou2d2ibi
